In [15]:
from google.colab import files

uploaded = files.upload()

Saving test.csv.zip to test.csv.zip


In [16]:
import shutil

shutil.unpack_archive(list(uploaded.keys())[0])

In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

torch.manual_seed(42)

Device: cpu


In [5]:
df = pd.read_csv("train.csv")

# df = pd.read_csv("test.csv")

print(df.shape)

(28000, 784)


In [20]:
df.sample(1)

,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
27312,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [11]:
df.head(1)

,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [21]:
X = df.iloc[:,1:].values.astype(np.float32)
y = df.iloc[:,0].values.astype(np.int64)

In [7]:
# reshape to (N,1,28,28)
X = X.reshape(-1,1,28,28)

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training:",X_train.shape)
print("Validation:",X_val.shape)

# Convert to Tensor

X_train = torch.tensor(X_train)
X_val = torch.tensor(X_val)

y_train = torch.tensor(y_train)
y_val = torch.tensor(y_val)

train_dataset = TensorDataset(X_train,y_train)
val_dataset = TensorDataset(X_val,y_val)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

Training: (33600, 1, 28, 28)
Validation: (8400, 1, 28, 28)


In [8]:
print(X_train.shape)
print(y_train.shape)
print(X_val.shape)
print(y_val.shape)

torch.Size([33600, 1, 28, 28])
torch.Size([33600])
torch.Size([8400, 1, 28, 28])
torch.Size([8400])


In [6]:
# #test

# X = df.iloc[:,:].values.astype(np.float32)

# X = X.reshape(-1,1,28,28)

# X = torch.tensor(X)

# test_dataset = TensorDataset(X)

# test_loader = DataLoader(
#     test_dataset,
#     batch_size=64,
#     shuffle=False
# )

# print(X.shape)

torch.Size([28000, 1, 28, 28])


In [8]:
#CNN model

class DigitCNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(1,32,3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32,32,3,padding=1),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64,64,3,padding=1),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Dropout(0.25)
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(64*7*7,256),
            nn.ReLU(),

            nn.Dropout(0.5),

            nn.Linear(256,128),
            nn.ReLU(),

            nn.Linear(128,10)

        )

    def forward(self,x):

        x = self.features(x)
        x = self.classifier(x)

        return x


model = DigitCNN().to(device)

print(model)

DigitCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU()
    (9): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Dropout(p=0.25, inplace=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_f

In [9]:
# Loss & Optimizer
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [11]:
# Training Function

def train(model,loader):

    model.train()

    running_loss = 0

    correct = 0

    total = 0

    for images,labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs,labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _,predicted = torch.max(outputs,1)

        total += labels.size(0)

        correct += (predicted==labels).sum().item()

    epoch_loss = running_loss/len(loader)

    accuracy = 100*correct/total

    return epoch_loss,accuracy

In [12]:
# Validation Function

def validate(model,loader):

    model.eval()

    running_loss = 0

    correct = 0

    total = 0

    with torch.no_grad():

        for images,labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs,labels)

            running_loss += loss.item()

            _,predicted = torch.max(outputs,1)

            total += labels.size(0)

            correct += (predicted==labels).sum().item()

    epoch_loss = running_loss/len(loader)

    accuracy = 100*correct/total

    return epoch_loss,accuracy

In [13]:
# Train Model

epochs = 4

best_accuracy = 0

for epoch in range(epochs):

    train_loss,train_acc = train(model,train_loader)

    val_loss,val_acc = validate(model,val_loader)

    print(f"Epoch {epoch+1}/{epochs}")

    print(f"Train Loss : {train_loss:.4f}")

    print(f"Train Accuracy : {train_acc:.2f}%")

    print(f"Validation Loss : {val_loss:.4f}")

    print(f"Validation Accuracy : {val_acc:.2f}%")

    print("-"*50)

    if val_acc > best_accuracy:

        best_accuracy = val_acc

        torch.save(model.state_dict(),"best_digit_model.pth")

print("Best Validation Accuracy:",best_accuracy)

Epoch 1/4
Train Loss : 0.2164
Train Accuracy : 93.04%
Validation Loss : 0.0551
Validation Accuracy : 98.29%
--------------------------------------------------
Epoch 2/4
Train Loss : 0.0702
Train Accuracy : 97.98%
Validation Loss : 0.0440
Validation Accuracy : 98.67%
--------------------------------------------------
Epoch 3/4
Train Loss : 0.0604
Train Accuracy : 98.18%
Validation Loss : 0.0370
Validation Accuracy : 99.02%
--------------------------------------------------
Epoch 4/4
Train Loss : 0.0447
Train Accuracy : 98.71%
Validation Loss : 0.0388
Validation Accuracy : 98.94%
--------------------------------------------------
Best Validation Accuracy: 99.02380952380952


In [10]:
# Load Best Model

model.load_state_dict(torch.load("best_digit_model.pth"))

model.eval()

print("Best model loaded.")

Best model loaded.


In [17]:
# image_idx = 1
# results = []

# with torch.no_grad():

#   for images in test_loader:

#       images = images[0].to(device)

#       outputs = model(images)

#       _,predicted = torch.max(outputs,1)

#       preds_list = predicted.cpu().tolist()

#       for pred in preds_list:
#         results.append([image_idx, pred])
#         image_idx += 1


In [22]:
# import csv

# csv_filename = "predictions.csv"
# with open(csv_filename, mode="w", newline="") as f:
#     writer = csv.writer(f)
#     writer.writerow(['ImageId', 'Label'])  # Write headers
#     writer.writerows(results)               # Write all data rows

# print(f"Saved predictions to {csv_filename}")

Saved predictions to predictions.csv
